In [126]:
import pandas as pd 
import numpy as np 
import plotly.express as px 
import re
import plotly.graph_objects as go 

In [127]:
bbb = pd.read_csv('data/ipl_dataset_2008-2026/ball_by_ball_data.csv')
matches = pd.read_csv('data/ipl_dataset_2008-2026/ipl_matches_data.csv', parse_dates=['match_date'])
players_data = pd.read_csv('data/ipl_dataset_2008-2026/players-data-updated.csv')
team_aliases = pd.read_csv('data/ipl_dataset_2008-2026/team_aliases.csv')
teams_data = pd.read_csv('data/ipl_dataset_2008-2026/teams_data.csv')
matches_2026 = pd.read_csv('data/ipl_matches.csv', parse_dates=['start_date'])

In [128]:
matches = matches.drop(columns=['gender', 'event_name', 'match_type', 'format', 'overs', 'team_type', 'match_number', 'season_id', 'balls_per_over'])

In [129]:
bbb['is_legal_ball'] = (~bbb['is_wide_ball']) & (~bbb['is_no_ball'])
bbb['is_legal_ball'] = bbb['is_legal_ball'].astype(int)
temp_series = pd.Series([])
for name, group in bbb.groupby(['match_id', 'innings']):
    temp_series = pd.concat([temp_series, group['is_legal_ball'].cumsum()])
bbb['innings_legal_balls'] = temp_series

C:\Users\Asus\AppData\Local\Temp\ipykernel_14284\786710980.py:5: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  temp_series = pd.concat([temp_series, group['is_legal_ball'].cumsum()])


In [130]:
bbb['is_dot'] = np.where((bbb['total_runs'] == 0), 1, 0)
bbb['is_boundary'] = bbb.apply(lambda x: 1 if (x['batter_runs'] == 4 or x['batter_runs'] == 6) else 0, axis=1)

In [131]:

# type Conversion
matches['season'] = matches['season'].astype('string').str[:4].astype(np.int32)
matches['bat_first_won'] = (matches['win_by_runs'] > 0).astype(int)
matches["toss_winner_won"] = (matches["toss_winner"] == matches["match_winner"]).astype(int)
matches['player_of_match'] = matches['player_of_match'].astype('Int32')
matches['city'] = matches['city'].astype('string')
matches['venue'] = matches['venue'].astype('category')
matches['toss_decision'] = matches['toss_decision'].astype('category')
matches['result'] = matches['result'].astype('category')

In [132]:
def fill_toss_decision(x):
    if pd.isna(x['toss_decision']):
        if not (pd.isna(x['win_by_runs']) & pd.isna(x['win_by_wickets'])):
            if x['toss_winner_won'] == 1:
                if pd.isna(x['win_by_runs']): # win by wickets
                    x['toss_decision'] = 'field'
                else : # win by runs 
                    x['toss_decision'] = 'bat'
            else :
                if pd.isna(x['win_by_runs']): # win by runs
                    x['toss_decision'] = 'bat'
                else : # win by wickets 
                    x['toss_decision'] = 'field'
        else :
            x['toss_decision'] = 'Not available'
    return x['toss_decision']
# find toss_decision by result of the match
matches['toss_decision'] = matches.apply(fill_toss_decision, axis=1).astype('category')
# fill missng values
matches['city'] = matches['city'].fillna('Unknown City')
matches['player_of_match'] = matches['player_of_match'].fillna(-1)

In [133]:
# Fix Unknown City
matches.loc[(matches['match_date'] == '2020-09-20') & (matches['city'] == 'Unknown City'), 'city'] = 'Dubai'
matches.loc[(matches['match_date'] == '2020-09-28') & (matches['city'] == 'Unknown City'), 'city'] = 'Dubai'
matches.loc[(matches['match_date'] == '2020-10-18') & (matches['city'] == 'Unknown City') & (matches['team1'] == 'Mumbai Indians'), 'city'] = 'Dubai'

# tie matches winner fixing
matches.loc[(matches['match_date'] == '2009-04-23') & (matches['venue'] == 'Cape Town'), 'match_winner'] = 134
matches.loc[(matches['match_date'] == '2010-03-21') & (matches['venue'] == 'Chennai'), 'match_winner'] = 494
matches.loc[(matches['match_date'] == '2013-04-07') & (matches['venue'] == 'Hyderabad'), 'match_winner'] = 2
matches.loc[(matches['match_date'] == '2013-04-16') & (matches['venue'] == 'Bangalore'), 'match_winner'] = 1
matches.loc[(matches['match_date'] == '2014-04-29') & (matches['venue'] == 'Abu Dhabi'), 'match_winner'] = 134
matches.loc[(matches['match_date'] == '2015-04-21') & (matches['venue'] == 'Ahmedabad'), 'match_winner'] = 494
matches.loc[(matches['match_date'] == '2017-04-29') & (matches['venue'] == 'Rajkot'), 'match_winner'] = 3
matches.loc[(matches['match_date'] == '2019-03-30') & (matches['venue'] == 'Delhi'), 'match_winner'] = 252
matches.loc[(matches['match_date'] == '2019-05-02') & (matches['venue'] == 'Mumbai'), 'match_winner'] = 3
matches.loc[(matches['match_date'] == '2020-09-20'), 'match_winner'] = 252
matches.loc[(matches['match_date'] == '2020-09-28'), 'match_winner'] = 1
matches.loc[(matches['match_date'] == '2020-10-18') & (matches['venue'] == 'Abu Dhabi'), 'match_winner'] = 6
matches.loc[(matches['match_date'] == '2020-10-18') & (matches['venue'].str.contains('Dubai', na=False)), 'match_winner'] = 494
matches.loc[(matches['match_date'] == '2021-04-25') & (matches['venue'] == 'Chennai'), 'match_winner'] = 252
matches.loc[(matches['match_date'] == '2025-04-16'), 'match_winner'] = 252
matches.loc[(matches['match_date'] == '2026-04-26'), 'match_winner'] = 6

In [134]:
bbb = bbb.rename(columns={'season_id':'season', 'wicket_kind':'dismissal_kind'})
bbb = bbb[['season', 'match_id', 'batter', 'bowler', 'non_striker', 'team_batting', 'team_bowling', 'over_number', 'ball_number', 'batter_runs', 'extras', 'total_runs', 'player_out', 'fielders_involved', 'is_wicket', 'is_boundary', 'is_dot', 'dismissal_kind', 'innings', 'innings_legal_balls']]

In [135]:
# filling missing values
bbb['player_out']        = bbb['player_out'].fillna('No One')
bbb['fielders_involved'] = bbb['fielders_involved'].fillna('No One')

# convert types
bbb['batter']       = bbb['batter'].astype('string')
bbb['bowler']       = bbb['bowler'].astype('string')
bbb['non_striker']  = bbb['non_striker'].astype('string')
bbb['player_out']   = bbb['player_out'].astype('string')

In [136]:
# false entry for player name "Unknown"
players_data.loc[len(players_data)] = [-1, 'Unknown','Unknown', 'Unknown', 'Unknown', 'Unknown'] 

In [137]:
# fill Unknown for ' ' entries 
players_data.loc[(players_data['bowl_style'] == ' '), 'bowl_style'] = 'Unknown'
players_data.loc[(players_data['field_pos'] == ' '), 'field_pos'] = 'Unknown'

# filling missing values
players_data['bowl_style'] = players_data['bowl_style'].fillna('Unknown') 
players_data['field_pos']  = players_data['field_pos'].fillna('Unknown') 

# convert types 
players_data['player_id'] = players_data['player_id'].astype(int)
players_data['player_name'] = players_data['player_name'].astype('string')
players_data['player_full_name'] = players_data['player_full_name'].astype('string') 
players_data['field_pos'] = players_data['field_pos'].astype('category') 
players_data['bowl_style'] = players_data['bowl_style'].astype('category') 
players_data['bat_style'] = players_data['bat_style'].astype('category') 

# fix duplicated  entries
players_data.loc[players_data['bowl_style'] == 'Right arm Fast medium', 'bowl_style'] = 'Right arm Fast Medium'


In [138]:
team_aliases['alias_name'] = team_aliases['alias_name'].astype('string')

In [139]:
teams_data['team_name'] = teams_data['team_name'].astype('string')

In [140]:
bbb_2026 = pd.read_csv('data/ipl_ballbyball_2026_sample.csv', parse_dates=['start_date'])
matches_2026 = pd.read_csv('data/ipl_matches.csv', parse_dates=['start_date'])

In [141]:
# delete the data of ipl 2026
matches = matches[~(matches['match_date'].dt.year == 2026)]
bbb = bbb[~(bbb['season'] == 2026)]

In [142]:
matches_2026 = matches_2026[['match_id', 'city', 'start_date', 'season', 'venue', 'toss_winner', 'team1', 'team2', 'toss_decision', 'winner', 'win_by_runs', 'win_by_wickets', 'result', 'player_of_match']].rename(columns={'start_date':'match_date', 'winner':'match_winner'})

In [143]:
matches_2026['season'] = matches_2026['season'].str.strip().str[:4]
matches_2026['season'] = matches_2026['season'].astype(int)

In [144]:
#select only 2026 matches
matches_2026 = matches_2026[matches_2026['season']==2026]
# correct wrong info
matches_2026.loc[(matches_2026['match_id'] == 1529281), 'match_winner'] = 'Kolkata Knight Riders'
matches_2026.loc[(matches_2026['match_id'] == 1529281), 'result'] = 'Win'
# filling missing values
matches_2026['result'] = matches_2026['result'].fillna('win')
matches_2026['player_of_match'] = matches_2026['player_of_match'].fillna('None')
matches_2026['match_winner'] = matches_2026['match_winner'].fillna('No Winner(Match Abandoned)')

In [145]:
# convert the type for merging with original data
matches_2026['season'] = matches_2026['season'].astype('string').str[:4].astype(np.int32)
matches_2026['bat_first_won'] = (matches_2026['win_by_runs'] > 0).astype(int)
matches_2026["toss_winner_won"] = (matches_2026["toss_winner"] == matches_2026["match_winner"]).astype(int)
matches_2026['player_of_match'] = matches_2026['player_of_match'].astype('string')
matches_2026['city'] = matches['city'].astype('string')
matches_2026['toss_winner'] = matches_2026['toss_winner'].astype('string')
matches_2026['team1'] = matches_2026['team1'].astype('string')
matches_2026['team2'] = matches_2026['team2'].astype('string')
matches_2026['match_winner'] = matches_2026['match_winner'].astype('string')
matches_2026['venue'] = matches_2026['venue'].astype('category')
matches_2026['toss_decision'] = matches_2026['toss_decision'].astype('category')
matches_2026['result'] = matches_2026['result'].astype('category')

In [146]:
matches = matches.merge(teams_data, left_on='toss_winner', right_on='team_id', how='left').drop(columns=['team_id', 'toss_winner']).rename(columns={'team_name':'toss_winner'})
matches = matches.merge(teams_data, left_on='team1', right_on='team_id', how='left').drop(columns=['team_id', 'team1']).rename(columns={'team_name':'team1'})
matches = matches.merge(teams_data, left_on='team2', right_on='team_id', how='left').drop(columns=['team_id', 'team2']).rename(columns={'team_name':'team2'})
matches = matches.merge(teams_data, left_on='match_winner', right_on='team_id', how='left').drop(columns=['team_id', 'match_winner']).rename(columns={'team_name':'match_winner'})
matches = matches.merge(players_data, left_on='player_of_match', right_on='player_id', how='left').drop(columns=['player_id', 'player_of_match', 'bat_style', 'bowl_style', 'field_pos', 'player_full_name']).rename(columns={'player_name':'player_of_match'})

In [147]:
matches = matches[['match_id', 'city', 'match_date', 'season', 'venue', 'toss_winner', 'team1', 'team2', 'toss_decision', 'match_winner', 'win_by_runs', 'win_by_wickets', 'result', 'player_of_match', 'bat_first_won', 'toss_winner_won']]

In [148]:
matches = pd.concat([matches, matches_2026])
matches = matches.reset_index(drop=True)
mask = (matches['match_date'].dt.year >= 2008) & (matches['match_date'].dt.year <= 2012)
matches.loc[mask] = matches[mask].replace(to_replace='Sunrisers Hyderabad', value='Deccan Chargers')

In [149]:
bbb_2026 = bbb_2026[['season', 'match_id', 'batter', 'bowler', 'non_striker', 'batting_team', 'bowling_team', 'over', 'ball_in_over', 'runs_batter', 'runs_extras', 'runs_total', 'player_dismissed', 'fielders', 'is_wicket', 'is_boundary', 'is_dot', 'dismissal_kind', 'innings', 'innings_legal_balls']].rename(columns={'batting_team':'team_batting', 'bowling_team':'team_bowling', 'over':'over_number', 'ball_in_over':'ball_number', 'runs_batter':'batter_runs', 'runs_extras':'extras', 'runs_total':'total_runs', 'player_dismissed':'player_out', 'fielders':'fielders_involved'})

In [150]:
bbb_2026['player_out']        = bbb_2026['player_out'].fillna('No One')
bbb_2026['fielders_involved'] = bbb_2026['fielders_involved'].fillna('No One')

In [151]:
bbb = bbb.merge(teams_data, left_on='team_batting', right_on='team_id', how='left').drop(columns=['team_batting', 'team_id']).rename(columns={'team_name':'team_batting'})
bbb = bbb.merge(teams_data, left_on='team_bowling', right_on='team_id', how='left').drop(columns=['team_bowling', 'team_id']).rename(columns={'team_name':'team_bowling'})
bbb['over_number'] = bbb['over_number']+1
bbb['ball_number'] = bbb['ball_number']+1

In [152]:
bbb = pd.concat([bbb, bbb_2026])

In [153]:
bbb['phase'] = bbb.apply(lambda x: "PowerPlay" if x['over_number'] > 0 and x['over_number'] < 7 else "Death" if x['over_number'] >= 16 else "Middle", axis=1)

In [154]:
idx = matches.groupby('season')['match_date'].idxmax()
finals = matches.loc[idx].sort_values('season', ascending=False).reset_index()

In [155]:
matches.info()
bbb.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1243 entries, 0 to 1242
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   match_id         1243 non-null   int64         
 1   city             1243 non-null   string        
 2   match_date       1243 non-null   datetime64[ns]
 3   season           1243 non-null   int32         
 4   venue            1243 non-null   object        
 5   toss_winner      1243 non-null   string        
 6   team1            1243 non-null   string        
 7   team2            1243 non-null   string        
 8   toss_decision    1243 non-null   category      
 9   match_winner     1243 non-null   string        
 10  win_by_runs      558 non-null    float64       
 11  win_by_wickets   660 non-null    float64       
 12  result           1243 non-null   object        
 13  player_of_match  1243 non-null   string        
 14  bat_first_won    1243 non-null   int64  

In [156]:
matches_comp = matches[matches["match_winner"].notna()]

played = pd.concat([matches_comp["team1"], matches_comp["team2"]]).value_counts()
won = matches_comp["match_winner"].value_counts()
win_pct = (won / played * 100).round(1).sort_values(ascending=False).reset_index().rename(columns={'index':'Team', 'count':'Win%'}).head(10)
px.bar(x=win_pct['Team'], y=win_pct['Win%'], labels={"x": "Team", "y": "Win %"})
# win_pct

In [157]:
final_match = bbb[bbb['match_id'] == 1535465]
temp = final_match.groupby(['innings', 'over_number'])['total_runs'].sum().reset_index()

progression = pd.DataFrame()
for name, group in temp.groupby('innings'):
    group['innings_runs'] = group['total_runs'].cumsum()
    progression = pd.concat([progression, group])
    
progression

,innings,over_number,total_runs,innings_runs
0,1,1,13,13
1,1,2,5,18
2,1,3,6,24
3,1,4,6,30
4,1,5,9,39
5,1,6,6,45
6,1,7,5,50
7,1,8,5,55
8,1,9,4,59
9,1,10,4,63


In [158]:
bbb['batter'] = bbb['batter'].astype('string')
bbb['bowler'] = bbb['bowler'].astype('string')
bbb['non_striker'] = bbb['non_striker'].astype('string')
bbb['player_out'] = bbb['player_out'].astype('string')
bbb['dismissal_kind'] = bbb['dismissal_kind'].astype('category')
bbb['team_batting'] = bbb['team_batting'].astype('string')
bbb['team_bowling'] = bbb['team_bowling'].astype('string')
bbb['phase'] = bbb['phase'].astype('category')

In [159]:
bbb.info()
bbb['phase'].value_counts()

<class 'pandas.core.frame.DataFrame'>
Index: 295732 entries, 0 to 17526
Data columns (total 21 columns):
 #   Column               Non-Null Count   Dtype   
---  ------               --------------   -----   
 0   season               295732 non-null  int64   
 1   match_id             295732 non-null  int64   
 2   batter               295732 non-null  string  
 3   bowler               295732 non-null  string  
 4   non_striker          295732 non-null  string  
 5   over_number          295732 non-null  int64   
 6   ball_number          295732 non-null  int64   
 7   batter_runs          295732 non-null  int64   
 8   extras               295732 non-null  int64   
 9   total_runs           295732 non-null  int64   
 10  player_out           295732 non-null  string  
 11  fielders_involved    295732 non-null  object  
 12  is_wicket            295732 non-null  int64   
 13  is_boundary          295732 non-null  int64   
 14  is_dot               295732 non-null  int64   
 15  dismis

phase
Middle       135655
PowerPlay     92904
Death         67173
Name: count, dtype: int64

In [160]:
bbb['over_number'].value_counts()

over_number
1     15788
2     15614
3     15416
4     15396
5     15382
6     15308
7     15251
8     15228
9     15190
10    15155
11    15084
12    15042
13    15024
14    14894
15    14787
16    14604
17    14348
18    13918
19    13065
20    11238
Name: count, dtype: int64

In [161]:
t = bbb[(bbb['match_id'] == 1535465) & (bbb['innings'] == 1)]
t.pivot(index='over_number', columns='ball_number', values='bowler')

ball_number,1,2,3,4,5,6,7,8,9
over_number,,,,,,,,,
1,JA Duffy,JA Duffy,JA Duffy,JA Duffy,JA Duffy,JA Duffy,JA Duffy,JA Duffy,JA Duffy
2,B Kumar,B Kumar,B Kumar,B Kumar,B Kumar,B Kumar,<NA>,<NA>,<NA>
3,JR Hazlewood,JR Hazlewood,JR Hazlewood,JR Hazlewood,JR Hazlewood,JR Hazlewood,<NA>,<NA>,<NA>
4,B Kumar,B Kumar,B Kumar,B Kumar,B Kumar,B Kumar,<NA>,<NA>,<NA>
5,JR Hazlewood,JR Hazlewood,JR Hazlewood,JR Hazlewood,JR Hazlewood,JR Hazlewood,<NA>,<NA>,<NA>
6,Rasikh Salam,Rasikh Salam,Rasikh Salam,Rasikh Salam,Rasikh Salam,Rasikh Salam,<NA>,<NA>,<NA>
7,KH Pandya,KH Pandya,KH Pandya,KH Pandya,KH Pandya,KH Pandya,<NA>,<NA>,<NA>
8,Rasikh Salam,Rasikh Salam,Rasikh Salam,Rasikh Salam,Rasikh Salam,Rasikh Salam,<NA>,<NA>,<NA>
9,KH Pandya,KH Pandya,KH Pandya,KH Pandya,KH Pandya,KH Pandya,<NA>,<NA>,<NA>


## Match Profile

## TEAM PROFILE

In [162]:
team = "Mumbai Indians"
team_matches = matches[(matches["team1"] == team) | (matches["team2"] == team)]
team_comp = team_comp = team_matches[team_matches["match_winner"].notna()]
season_record = team_comp.groupby("season").apply( # type: ignore
    lambda g: round((g["match_winner"] == team).mean() * 100, 1), include_groups=False # type: ignore
)
season_record

season
2008    50.0
2009    38.5
2010    68.8
2011    62.5
2012    58.8
2013    68.4
2014    46.7
2015    62.5
2016    50.0
2017    70.6
2018    42.9
2019    62.5
2020    68.8
2021    50.0
2022    28.6
2023    56.2
2024    28.6
2025    56.2
2026    28.6
dtype: float64

In [163]:
matches_temp = matches[(matches['team1'] == 'Mumbai Indians') | (matches['team2'] == 'Mumbai Indians')].copy()
def fill(x):
    if pd.isna(x['win_by_runs']) & pd.isna(x['win_by_wickets']):
        return "N/A"
    elif pd.isna(x['win_by_runs']):
        return str(int(x['win_by_wickets'])) + " Wickets"
    else:
        return str(int(x['win_by_runs'])) + " Runs"
matches_temp['margin'] = matches_temp.apply(fill, axis=1)
matches_temp

,match_id,city,match_date,season,venue,toss_winner,team1,team2,toss_decision,match_winner,win_by_runs,win_by_wickets,result,player_of_match,bat_first_won,toss_winner_won,margin
2,1082592,Pune,2017-04-06,2017,Maharashtra Cricket Association Stadium,Rising Pune Supergiant,Rising Pune Supergiant,Mumbai Indians,field,Rising Pune Supergiant,NaN,7.0,win,SPD Smith,0,1,7 Wickets
7,1082597,Mumbai,2017-04-09,2017,Wankhede Stadium,Mumbai Indians,Mumbai Indians,Kolkata Knight Riders,field,Mumbai Indians,NaN,4.0,win,N Rana,0,1,4 Wickets
10,1082600,Mumbai,2017-04-12,2017,Wankhede Stadium,Mumbai Indians,Mumbai Indians,Sunrisers Hyderabad,field,Mumbai Indians,NaN,4.0,win,JJ Bumrah,0,1,4 Wickets
12,1082602,Bangalore,2017-04-14,2017,M Chinnaswamy Stadium,Mumbai Indians,Royal Challengers Bengaluru,Mumbai Indians,field,Mumbai Indians,NaN,4.0,win,KA Pollard,0,1,4 Wickets
16,1082606,Mumbai,2017-04-16,2017,Wankhede Stadium,Mumbai Indians,Mumbai Indians,Gujarat Lions,field,Mumbai Indians,NaN,6.0,win,N Rana,0,1,6 Wickets
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1215,1529290,Centurion,2026-05-04,2026,"Wankhede Stadium, Mumbai",Mumbai Indians,Lucknow Super Giants,Mumbai Indians,field,Mumbai Indians,NaN,6.0,win,RD Rickelton,0,1,6 Wickets
1222,1529297,Chennai,2026-05-10,2026,Shaheed Veer Narayan Singh International Stadi...,Royal Challengers Bengaluru,Mumbai Indians,Royal Challengers Bengaluru,field,Royal Challengers Bengaluru,NaN,2.0,win,B Kumar,0,1,2 Wickets
1226,1529301,Delhi,2026-05-14,2026,"Himachal Pradesh Cricket Association Stadium, ...",Mumbai Indians,Punjab Kings,Mumbai Indians,field,Mumbai Indians,NaN,6.0,win,Tilak Varma,0,1,6 Wickets
1233,1529308,Chennai,2026-05-20,2026,"Eden Gardens, Kolkata",Kolkata Knight Riders,Mumbai Indians,Kolkata Knight Riders,field,Kolkata Knight Riders,NaN,4.0,win,MK Pandey,0,1,4 Wickets


## PLAYER PROFILE

In [164]:
# jsut for practice
t = bbb['fielders_involved'].value_counts().reset_index()
t[t['fielders_involved'].str.len() > 21]


,fielders_involved,count
170,['Rahmanullah Gurbaz'],17
229,['Nithish Kumar Reddy'],12
300,"['MK Pandey', 'RV Uthappa']",8
442,['JPR Scantlebury-Searles'],4
457,"['RA Jadeja', 'MS Dhoni']",4
...,...,...
1279,"['W Jaffer', 'MV Boucher']",1
1280,"['V Sehwag', 'R Bhatia']",1
1281,"['DPMD Jayawardene', 'KC Sangakkara']",1
1282,"['MS Dhoni', 'Joginder Sharma']",1


In [165]:
# find any rows where a name itself might contains "'"
suspects = t['fielders_involved'][t['fielders_involved'].str.contains(r"\w'\w", regex=True, na=False)]
print(suspects) # W O'Rourke

1123    ["W O'Rourke"]
Name: fielders_involved, dtype: object


In [166]:
# correction in bbb['fielders_involved]
bbb['fielders_involved'] = bbb['fielders_involved'].str.strip().str.strip('[').str.strip(']').str.strip().str.replace(', ', '|').str.replace("'", "")
bbb.loc[(bbb['fielders_involved'] == 'No One'), 'fielders_involved'] = 'None'
bbb.loc[bbb['fielders_involved'] == "W ORourke", 'fielders_involved'] = "W O'Rourke"

In [167]:
# bbb.info()
# # bbb.head()

In [168]:
batter_stat = bbb.groupby(['batter', 'match_id']).agg(
    bat_runs=('batter_runs', 'sum'),
    bat_balls=('batter_runs', 'count')    
).reset_index()
only_fours = bbb.groupby(['batter', 'match_id']).apply( # type: ignore
    (lambda g: (g['batter_runs'] == 4).sum()) ,include_groups=False # type: ignore
).reset_index()
only_sixes = bbb.groupby(['batter', 'match_id']).apply( # type: ignore
    (lambda g: (g['batter_runs'] == 6).sum()) ,include_groups=False # type: ignore
).reset_index()
dismissed = bbb.groupby(['batter', 'match_id']).apply( # type: ignore
    (lambda g: (g['player_out'] == g.name[0]).sum()) ,include_groups=False # type: ignore
).reset_index()  # type : ignore

In [169]:
batter_stat = batter_stat.merge(only_fours, on=['batter', 'match_id'], how='left').rename(columns={0:'bat_fours'}).merge(only_sixes, on=['batter', 'match_id'], how='left').rename(columns={0:'bat_sixes'}).merge(dismissed, on=['batter', 'match_id'], how='left').rename(columns={0:'dismissed'})

In [170]:
batter_stat['bat_strike_rate'] = (batter_stat['bat_runs'] / batter_stat['bat_balls'])*100
bolwer_stat = bbb.groupby(['bowler', 'match_id']).agg(
    bowl_balls=('total_runs', 'count'),
    bowl_runs_conceded=('total_runs', 'sum'),
    bowl_wickets=('is_wicket', 'sum')
).reset_index()
bolwer_stat['bowl_economy'] = np.round(((bolwer_stat['bowl_runs_conceded'] / bolwer_stat['bowl_balls']) * 6), 2)
bolwer_stat

,bowler,match_id,bowl_balls,bowl_runs_conceded,bowl_wickets,bowl_economy
0,A Ashish Reddy,548329,12,29,0,14.50
1,A Ashish Reddy,548341,24,32,2,8.00
2,A Ashish Reddy,548346,14,11,1,4.71
3,A Ashish Reddy,548348,19,32,1,10.11
4,A Ashish Reddy,548352,13,16,1,7.38
...,...,...,...,...,...,...
14695,Zeeshan Ansari,1473478,22,36,1,9.82
14696,Zeeshan Ansari,1473480,18,28,0,9.33
14697,Zeeshan Ansari,1473488,25,42,2,10.08
14698,Zeeshan Ansari,1473492,18,30,1,10.00


In [171]:

fielding_stat = bbb[bbb['is_wicket'] == 1]
# fielding_stat[fielding_stat['dismissal_kind'] == 'run out']['fielders_involved'].str.split(r"|").str[2].notna().sum()
players_stat = batter_stat.merge(bolwer_stat, left_on=['batter', 'match_id'], right_on=['bowler', 'match_id'], how='outer')
# fielding_stat.groupby(['match_id']).agg(
    # run_outs=(),
    # catches=(),
    # stumpings=()
# )
players_stat['player'] = players_stat.apply(lambda x: x['batter'] if not pd.isna(x['batter']) else x['bowler'], axis=1)
players_stat = players_stat.merge(matches, on='match_id', how='left')
players_stat = players_stat[['match_id', 'player', 'season', 'match_date', 'bat_runs', 'bat_balls', 'bat_fours', 'bat_sixes', 'bat_strike_rate', 'dismissed', 'bowl_balls', 'bowl_runs_conceded', 'bowl_wickets', 'bowl_economy']]

run_outs = fielding_stat[fielding_stat['dismissal_kind'] == 'run out'].groupby(['match_id']).agg(
    run_outs=('fielders_involved', lambda x: ','.join(x))
)
caughts = fielding_stat[fielding_stat['dismissal_kind'] == 'caught'].groupby(['match_id']).agg(
    catches=('fielders_involved', lambda x: ','.join(x))
)

stumpings = fielding_stat[fielding_stat['dismissal_kind'] == 'stumped'].groupby(['match_id']).agg(
    stumpings=('fielders_involved', lambda x: ','.join(x))
)

caught_and_bowled = fielding_stat[fielding_stat['dismissal_kind'] == 'caught and bowled'].groupby(['match_id']).agg(
    run_outs=('fielders_involved', lambda x: ','.join(x))
)

# add run_outs
players_stat = players_stat.merge(run_outs, on='match_id', how='left')
players_stat['run_outs'] = players_stat['run_outs'].str.replace("|", ",")
players_stat['run_outs'] = players_stat.apply(
    lambda x: x['run_outs'].count(x['player']) if pd.notna(x['run_outs']) else 0, axis=1
)

# add catches
players_stat = players_stat.merge(caughts, on='match_id', how='left')
players_stat['catches'] = players_stat.apply(
    lambda x: x['catches'].count(x['player']) if pd.notna(x['catches']) else 0, axis=1
)

# add stumpings
players_stat = players_stat.merge(stumpings, on='match_id', how='left')
players_stat['stumpings'] = players_stat.apply(
    lambda x: x['stumpings'].count(x['player']) if pd.notna(x['stumpings']) else 0, axis=1
)

players_stat[players_stat['stumpings'] > 3]['stumpings']

Series([], Name: stumpings, dtype: int64)

In [172]:
t = players_stat.merge(bbb, on='match_id', how='left')

In [173]:
p = t[(t['player'] == 'A Ashish Reddy')]

In [174]:
matches_custom = matches.copy()
matches_custom['toss_loser'] = np.where((matches_custom['toss_winner'] == matches_custom['team1']), matches_custom['team2'], matches_custom['team1'])
def find_team(group):
    if group[group['player'] == group['batter']].shape[0] > 0:
        if group[group['player'] == group['batter']]['innings'].iloc[0] % 2 != 0:
            if matches_custom.loc[matches_custom['match_id'] == group.name, 'toss_decision'].iloc[0] == 'bat': # type: ignore
                group['team'] = matches_custom.loc[matches_custom['match_id'] == group.name, 'toss_winner'].iloc[0] # type: ignore
            else :
                group['team'] = matches_custom.loc[matches_custom['match_id'] == group.name, 'toss_loser'].iloc[0] # type: ignore
        else :
            if matches_custom.loc[matches_custom['match_id'] == group.name, 'toss_decision'].iloc[0] == 'bat': # type: ignore
                group['team'] = matches_custom.loc[matches_custom['match_id'] == group.name, 'toss_loser'].iloc[0] # type: ignore
            else :
                group['team'] = matches_custom.loc[matches_custom['match_id'] == group.name, 'toss_winner'].iloc[0] # type: ignore
    else :
        if group[group['player'] == group['bowler']]['innings'].iloc[0] % 2 != 0:
            if matches_custom.loc[matches_custom['match_id'] == group.name, 'toss_decision'].iloc[0] == 'field': # type: ignore
                group['team'] = matches_custom.loc[matches_custom['match_id'] == group.name, 'toss_winner'].iloc[0] # type: ignore
            else :
                group['team'] = matches_custom.loc[matches_custom['match_id'] == group.name, 'toss_loser'].iloc[0] # type: ignore
        else :
            if matches_custom.loc[matches_custom['match_id'] == group.name, 'toss_decision'].iloc[0] == 'field': # type: ignore
                group['team'] = matches_custom.loc[matches_custom['match_id'] == group.name, 'toss_loser'].iloc[0] # type: ignore
            else :
                group['team'] = matches_custom.loc[matches_custom['match_id'] == group.name, 'toss_winner'].iloc[0] # type: ignore
    return group

result = t.groupby(['match_id']).apply(find_team, include_groups=False).reset_index().drop_duplicates(subset=['match_id', 'player']).reset_index(drop=True) # type:ignore

In [175]:
players_stat = players_stat.merge(result, on=['match_id', 'player'], how='left').merge(matches[['match_id', 'team1', 'team2']], on='match_id', how='left')

In [176]:
# players_stat.info()

In [177]:
players_stat['opponent'] = np.where((players_stat['team'] == players_stat['team1']), players_stat['team2'], players_stat['team1'])
players_stat = players_stat[['match_id', 'season', 'match_date_x', 'team', 'opponent', 'player', 'bat_runs_x', 'bat_balls_x', 'bat_fours_x', 'bat_sixes_x', 'bat_strike_rate_x', 'dismissed_x', 'bowl_balls_x', 'bowl_runs_conceded_x', 'bowl_wickets_x', 'bowl_economy_x','run_outs_x', 'catches_x', 'stumpings_x']].rename(columns={'match_date_x':'match_date', 'bat_runs_x':'bat_runs', 'bat_balls_x':'bat_balls', 'bat_fours_x':'bat_fours', 'bat_sixes_x':'bat_sixes', 'bat_strike_rate_x':'bat_strike_rate', 'dismissed_x':'dismissed', 'bowl_balls_x':'bowl_balls', 'bowl_runs_conceded_x':'bowl_runs_conceded', 'bowl_wickets_x':'bowl_wickets', 'bowl_economy_x':'bowl_economy', 'team_x':'team', 'run_outs_x':'run_outs', 'catches_x':'catches', 'stumpings_x':'stumpings'})


In [178]:
def role_find(g):
    g = g.copy()
    bats = (g['bat_runs'].notna().sum() / g.shape[0]) * 100
    balls = (g['bowl_balls'].notna().sum() / g.shape[0]) * 100
    if 0 <= abs(bats - balls) <= 25:
        avg = ((g['bat_runs'].sum() / g.shape[0]) * 100)
        if avg > 17:
            g['estimated_role'] = 'All Rounder'
        else :
            g['estimated_role'] = 'Bowler'
    elif bats - balls > 0:
        g['estimated_role'] = 'Batsman'
    else :
        g['estimated_role'] = 'Bowler'
    return g
players_stat = players_stat.groupby(['player']).apply(role_find, include_groups=False) # type: ignore
players_stat = players_stat.reset_index()

In [179]:
players_stat = players_stat.fillna(0)

In [180]:
def correct_team(g):
    g = g.copy()
    t1 = g['team'].value_counts().reset_index()
    t2 = g['opponent'].value_counts().reset_index()
    t = t1.merge(t2, left_on='team', right_on='opponent', how='outer')
    t['team'] = np.where(pd.isna(t['team']), t['opponent'], t['team'])
    t['count_x'] = t['count_x'].fillna(0)
    t['count_y'] = t['count_y'].fillna(0)
    t['count'] = t['count_x'] + t['count_y']
    t = t.drop(columns=['count_x','count_y', 'opponent']).sort_values(by='count', ascending=False)['team'].iloc[0]
    g['team'] = t
    return g
    
result = players_stat.groupby(['player', 'season']).apply(correct_team, include_groups=False) # type: ignore
result = result.reset_index()

In [181]:
result['opponent'] = np.where((result['team'] == players_stat['opponent']), players_stat['team'], players_stat['opponent'])

In [182]:
players_stat['team'] = result['team']
players_stat['opponent'] = result['opponent']
players_stat.loc[(players_stat['match_id'] == 598009) & (players_stat['player'] == 'Harmeet Singh'), ['player', 'team', 'opponent']] = ['Harmeet Singh Bansal', 'Rajasthan Royals', 'Pune Warriors']
players_stat.loc[(players_stat['match_id'] == 598009) & (players_stat['player'] == 'Harmeet Singh Bansal')]

,player,level_1,match_id,season,match_date,team,opponent,bat_runs,bat_balls,bat_fours,...,bat_strike_rate,dismissed,bowl_balls,bowl_runs_conceded,bowl_wickets,bowl_economy,run_outs,catches,stumpings,estimated_role
7650,Harmeet Singh Bansal,7650,598009,2013,2013-04-11,Rajasthan Royals,Pune Warriors,0.0,0.0,0.0,...,0.0,0.0,25.0,34.0,1.0,8.16,0,0,0,Bowler


In [183]:
players_stat['bat_runs'] = players_stat['bat_runs'].astype('Int32')
players_stat['bat_balls'] = players_stat['bat_balls'].astype('Int32')
players_stat['bat_fours'] = players_stat['bat_fours'].astype('Int32')
players_stat['bat_sixes'] = players_stat['bat_sixes'].astype('Int32')
players_stat['dismissed'] = players_stat['dismissed'].astype('Int32')
players_stat['bowl_balls'] = players_stat['bowl_balls'].astype('Int32')
players_stat['bowl_runs_conceded'] = players_stat['bowl_runs_conceded'].astype('Int32')
players_stat['bowl_wickets'] = players_stat['bowl_wickets'].astype('Int32')
players_stat['bat_strike_rate'] = np.round(players_stat['bat_strike_rate'], 2).astype('Float32')
players_stat['bowl_economy'] = np.round(players_stat['bowl_economy'], 2).astype('Float32')

In [184]:
pp = pd.read_csv('data/ipl_dataset_2008-2026/players-data-updated.csv')

In [185]:
not_players = set(players_stat['player']) - set(pp['player_name'])
players_stat = players_stat[~players_stat['player'].isin(not_players)]
players_stat = players_stat.merge(pp, left_on='player', right_on='player_name')

In [186]:
players_stat = players_stat.drop(columns=['level_1', 'player_id', 'player_name'])

In [187]:
players_stat['half_century'] = players_stat['bat_runs'] >= 50
players_stat['century'] = players_stat['bat_runs'] >= 100

players_stat['half_century'] = players_stat['half_century'].astype('Int32')
players_stat['century'] = players_stat['century'].astype('Int32')

In [188]:
players_stat.to_csv('data/ipl_dataset_2008-2026/players_matches_stat.csv')
# players_stat = pd.read_csv('data/ipl_dataset_2008-2026/players_matches_stat.csv')

In [189]:
# players_stat = players_stat.drop(columns=['Unnamed: 0'])


In [190]:
player_a = 'V Kohli'
player_b = 'MS Dhoni'
players = set([player_a, player_b])
players_data = players_stat[players_stat['player'].isin(players)]

players_data = players_data.groupby('player').agg(
    Matches=('match_id', 'count'),
    Runs=('bat_runs', 'sum'),
    Avg_Runs=('bat_runs', 'mean'),
    Avg_Strike_Rate=('bat_strike_rate', 'mean'),
    Highest_Runs=('bat_runs', 'max'),
    Wickets=('bowl_wickets', 'sum'),
    Avg_Wickets_Per_Match=('bowl_wickets', 'mean'),
    Avg_Bowl_Econ=('bowl_economy', 'mean'),
    Best_bowling_w=('bowl_wickets', 'max'),
    Best_bowling_e=('bowl_economy', 'min'),
    Catches=('catches', 'sum')
)
players_data['best_bowling'] = (players_data['Best_bowling_w'].astype('string') + "/" + players_data['Best_bowling_e'].astype('string'))

players_data['Avg_Bowl_Econ'] = np.round(players_data['Avg_Bowl_Econ'], 2)
players_data['Avg_Runs'] = np.round(players_data['Avg_Runs'], 2)
players_data['Avg_Strike_Rate'] = np.round(players_data['Avg_Strike_Rate'], 2)
players_data['Avg_Wickets_Per_Match'] = np.round(players_data['Avg_Wickets_Per_Match'], 2)

players_data = players_data.reset_index().drop(columns=['Best_bowling_w', 'Best_bowling_e'])
temp = players_data[['player', 'Matches', 'Runs', 'Wickets', 'Catches']].melt(id_vars='player').sort_values(by='variable')

px.bar(temp, x='variable', y='value', barmode='group', color='player', category_orders={
    'variable':['Matches', 'Runs', 'Wickets', 'Catches']
})

players_data = players_data.set_index('player').stack().swaplevel(0).unstack()
players_data.index.name='Parameters'
players_data

player,MS Dhoni,V Kohli
Parameters,,
Matches,241,276
Runs,5439,9346
Avg_Runs,22.57,33.86
Avg_Strike_Rate,126.620003,116.82
Highest_Runs,84,113
Wickets,0,5
Avg_Wickets_Per_Match,0.0,0.02
Avg_Bowl_Econ,0.0,0.81
Catches,139,125


In [191]:
players_stat

,player,match_id,season,match_date,team,opponent,bat_runs,bat_balls,bat_fours,bat_sixes,...,run_outs,catches,stumpings,estimated_role,bat_style,bowl_style,field_pos,player_full_name,half_century,century
0,A Ashish Reddy,548329,2012,2012-05-10,Deccan Chargers,Delhi Capitals,0,0,0,0,...,0,0,0,All Rounder,Right hand Bat,Right arm Medium,,Ammana Ashish Reddy,0,0
1,A Ashish Reddy,548341,2012,2012-04-26,Deccan Chargers,Pune Warriors,0,0,0,0,...,0,0,0,All Rounder,Right hand Bat,Right arm Medium,,Ammana Ashish Reddy,0,0
2,A Ashish Reddy,548346,2012,2012-04-29,Deccan Chargers,Mumbai Indians,10,10,0,1,...,0,0,0,All Rounder,Right hand Bat,Right arm Medium,,Ammana Ashish Reddy,0,0
3,A Ashish Reddy,548348,2012,2012-05-01,Deccan Chargers,Pune Warriors,0,0,0,0,...,0,0,0,All Rounder,Right hand Bat,Right arm Medium,,Ammana Ashish Reddy,0,0
4,A Ashish Reddy,548352,2012,2012-05-04,Deccan Chargers,Chennai Super Kings,3,3,0,0,...,0,0,0,All Rounder,Right hand Bat,Right arm Medium,,Ammana Ashish Reddy,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26668,Zeeshan Ansari,1473478,2025,2025-04-23,Sunrisers Hyderabad,Mumbai Indians,0,0,0,0,...,0,0,0,Bowler,Right hand Bat,Legbreak Googly,NaN,Zeeshan Ansari,0,0
26669,Zeeshan Ansari,1473480,2025,2025-04-25,Sunrisers Hyderabad,Chennai Super Kings,0,0,0,0,...,0,0,0,Bowler,Right hand Bat,Legbreak Googly,NaN,Zeeshan Ansari,0,0
26670,Zeeshan Ansari,1473488,2025,2025-05-02,Sunrisers Hyderabad,Gujarat Titans,0,0,0,0,...,0,0,0,Bowler,Right hand Bat,Legbreak Googly,NaN,Zeeshan Ansari,0,0
26671,Zeeshan Ansari,1473492,2025,2025-05-05,Sunrisers Hyderabad,Delhi Capitals,0,0,0,0,...,1,0,0,Bowler,Right hand Bat,Legbreak Googly,NaN,Zeeshan Ansari,0,0


In [192]:
temp = players_stat.groupby(['player', 'estimated_role']).agg(
    strike_rate=("bat_strike_rate", "mean"),
    avg_runs=("bat_runs", "mean"),
    experience=('match_id', 'count'),
    t=("player_full_name", "first")
).reset_index()
temp = temp[temp['experience'] > 5]
px.scatter(temp, x="strike_rate", y="avg_runs", hover_name="t", color="estimated_role", size="experience")

In [193]:
matches['team1'].unique()

<StringArray>
['Royal Challengers Bengaluru',         'Sunrisers Hyderabad',
      'Rising Pune Supergiant',               'Gujarat Lions',
                'Punjab Kings',              'Mumbai Indians',
       'Kolkata Knight Riders',              'Delhi Capitals',
            'Rajasthan Royals',         'Chennai Super Kings',
        'Lucknow Super Giants',              'Gujarat Titans',
             'Deccan Chargers',        'Kochi Tuskers Kerala',
               'Pune Warriors']
Length: 15, dtype: string

In [194]:
bbb.info()

<class 'pandas.core.frame.DataFrame'>
Index: 295732 entries, 0 to 17526
Data columns (total 21 columns):
 #   Column               Non-Null Count   Dtype   
---  ------               --------------   -----   
 0   season               295732 non-null  int64   
 1   match_id             295732 non-null  int64   
 2   batter               295732 non-null  string  
 3   bowler               295732 non-null  string  
 4   non_striker          295732 non-null  string  
 5   over_number          295732 non-null  int64   
 6   ball_number          295732 non-null  int64   
 7   batter_runs          295732 non-null  int64   
 8   extras               295732 non-null  int64   
 9   total_runs           295732 non-null  int64   
 10  player_out           295732 non-null  string  
 11  fielders_involved    295732 non-null  object  
 12  is_wicket            295732 non-null  int64   
 13  is_boundary          295732 non-null  int64   
 14  is_dot               295732 non-null  int64   
 15  dismis

In [195]:

team = "Chennai Super Kings"
team_data_batting = bbb[(bbb['season'] > 2025) & (bbb['team_batting'] == team)]
team_data_bowling = bbb[(bbb['season'] > 2025) & (bbb['team_bowling'] == team)]
all_players1 = team_data_batting['batter'].value_counts().reset_index()
all_players2 = team_data_bowling['bowler'].value_counts().reset_index()

all_players = all_players1.merge(all_players2, left_on='batter', right_on='bowler', how='outer')
all_players['player'] = np.where(all_players['batter'].isna(), all_players['bowler'], all_players['batter'])
all_players['count'] = all_players['count_x'].fillna(0) + all_players['count_y'].fillna(0)
all_players = all_players.drop(columns=['count_x', 'count_y', 'batter', 'bowler'])
all_players = all_players.sort_values('count', ascending=False)
all_players = all_players.merge(players_stat[['player', 'estimated_role']], on='player', how='inner').drop_duplicates(subset=['player', 'count'])
all_players = all_players.sort_values(by=['estimated_role', 'count'], ascending=False).reset_index(drop=True)
batting_players = all_players[all_players['estimated_role'] == "Batsman"]
bowling_players = all_players[all_players['estimated_role'] == "Bowler"]
all_rounder_players = all_players[all_players['estimated_role'] == "All Rounder"]
bowling_players = bowling_players.merge(players_stat[['player', 'bowl_style', 'player_full_name']], on='player', how='left').drop_duplicates(subset=['player', 'count'])
batting_players = batting_players.merge(players_stat[['player', 'bat_style', 'player_full_name']], on='player', how='left').drop_duplicates(subset=['player', 'count'])
all_rounder_players = all_rounder_players.merge(players_stat[['player', 'bat_style', 'bowl_style', 'player_full_name']], on='player', how='left').drop_duplicates(subset=['player', 'count'])
batting_players = batting_players.iloc[:7, :]
bowling_players = bowling_players.iloc[:7, :]
all_rounder_players = all_rounder_players.iloc[:5, :]
bbb['team_batting'].unique()
all_players

,player,count,estimated_role
0,A Kamboj,355,Bowler
1,Noor Ahmad,338,Bowler
2,J Overton,267,Bowler
3,Mukesh Choudhary,180,Bowler
4,Gurjapneet Singh,159,Bowler
5,AJ Hosein,154,Bowler
6,KK Ahmed,106,Bowler
7,SH Johnson,72,Bowler
8,MJ Henry,71,Bowler
9,RD Chahar,25,Bowler
